# Chapter 1: Similarity and Distance (Solutions)


**Agenda**

☕ · 📏 · 🚕 · 🧭 · 🔍 · ⚖️ · 🧩 · 🗺️ · 📊 · 📐 · 💡 · 🏁

**Next steps (take it from here):** 🌍 · 🎯 · 🔥 · 🧬

> **Tip:** Run cells top to bottom. Later cells depend on earlier ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from checks import (check_euclidean, check_manhattan, check_cosine_sim,
                    check_cosine_dist, check_mahalanobis, check_pairwise,
                    check_zscore, check_minmax)


def tufte_axis(ax):
    """Remove spines, keep only outward ticks on left and bottom."""
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis='both', which='both', direction='out',
                   length=5, width=1.2, colors='black',
                   top=False, right=False)

## ☕ The Coffee Lab

Imagine you work in a coffee quality lab. You have measured 20 samples from four brewing methods across three dimensions: **acidity** (pH), **bitterness** (scale 1–10), and **brew strength** (mg/mL).

Your task: figure out which coffees are similar — and what "similar" even means when features live on very different scales.

> As you look at the data below, ask yourself: which feature will dominate a naive distance calculation, and why?

<details><summary>Thought</summary>

Brew strength ranges from 7 to 24 mg/mL (a spread of ~17), while acidity spans only ~2 pH units and bitterness ~7 units. Any distance metric that sums differences will be dominated by brew strength simply because its numbers are bigger.
</details>

In [ ]:
# Load the dataset
df = pd.read_csv("coffee_samples.csv")
print(df.to_string(index=False))

In [ ]:
# Extract the numeric features and labels
features = df[["acidity", "bitterness", "brew_strength"]].values
labels = df["variety"].values
sample_ids = df["sample_id"].values

# Color map for varieties
variety_colors = {"cold_brew": "tab:blue", "drip": "tab:orange",
                  "espresso": "tab:red", "latte": "tab:green"}
colors = [variety_colors[v] for v in labels]

Let's look at the data. We plot two projections: acidity vs. bitterness and acidity vs. brew strength.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.scatter(features[:, 0], features[:, 1], c=colors, s=20, linewidths=0)
ax1.set_xlabel('Acidity (pH)')
ax1.set_ylabel('Bitterness')
tufte_axis(ax1)

ax2.scatter(features[:, 0], features[:, 2], c=colors, s=20, linewidths=0)
ax2.set_xlabel('Acidity (pH)')
ax2.set_ylabel('Brew Strength (mg/mL)')
tufte_axis(ax2)

# Legend
for variety, color in variety_colors.items():
    ax2.scatter([], [], c=color, s=20, label=variety)
ax2.legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

## Distance Metrics from Scratch

Now let's quantify similarity. We pick two samples from the dataset — a cold brew and an espresso — and measure how far apart they are using four different metrics.

Each function takes two 1‑D arrays and returns a single number. Implement them using only basic NumPy operations — no `scipy`, no shortcuts.

In [ ]:
# Pick two samples: a cold brew and an espresso
x = features[0]    # S01: cold_brew  (acidity=6.2, bitterness=2.1, brew_strength=23.0)
y = features[11]   # S12: espresso   (acidity=4.3, bitterness=8.5, brew_strength=23.0)

print(f'S01 (cold brew): {x}')

print(f'S12 (espresso):  {y}')

### 📏 Euclidean Distance

> Euclidean distance assumes all dimensions are equally important and 
measured in the same units. Look at our two coffee samples: acidity 
differs by ~1.9 pH, bitterness by ~6.4, and brew strength is identical. 
Which of these differences will matter most in the sum of squares, 
and is that actually the most meaningful difference?

<details><summary>Thought</summary>

The bitterness gap (6.4) contributes the most to the squared sum 
(41.0), followed by acidity (3.6). Brew strength contributes nothing 
here because these two samples happen to match on that feature. But 
across other pairs (e.g. cold brew vs. latte) brew strength would 
dominate everything else.
</details>

Revisit the lecture notes for the formula, then translate it into NumPy.

Useful operations: `np.sum()`, `np.sqrt()`, and the `**` operator for powers.

In [ ]:
def euclidean_distance(x, y):
    """Return the Euclidean distance between two 1-D arrays."""
    return np.sqrt(np.sum((x - y) ** 2))


check_euclidean(euclidean_distance, x, y)

### 🚕 Manhattan Distance

> Euclidean distance squares each coordinate difference before summing. Manhattan distance takes the absolute value instead. How does squaring amplify the effect of a single large difference compared to taking the absolute value?

<details><summary>Thought</summary>

Squaring is superlinear: a difference of 10 contributes 100, while two differences of 5 contribute only 50. So Euclidean is more sensitive to a single outlier dimension, whereas Manhattan treats each dimension's contribution proportionally.
</details>

Check the lecture notes for the formula.

Useful operation: `np.abs()` for absolute values.

In [ ]:
def manhattan_distance(x, y):
    """Return the Manhattan distance between two 1-D arrays."""
    return np.sum(np.abs(x - y))


check_manhattan(manhattan_distance, x, y)

### 🧭 Cosine Similarity & Distance

> The dot product of two vectors encodes both their magnitudes and their angle. The denominator in cosine similarity divides out the magnitudes. Cold brew and espresso have similar brew strength but very different bitterness. Does cosine similarity see them as close (both have large brew-strength components pulling them in a similar direction) or as far apart?

<details><summary>Thought</summary>

It depends on the balance of components. Both samples have a dominant brew-strength component (~22–23), which pulls the vectors in a similar direction. But the bitterness gap (2.1 vs. 8.2) rotates them apart. Cosine similarity will be high but not 1 — the shared large brew-strength component masks some of the bitterness difference.
</details>

The cosine **distance** is simply `1 - similarity`.

Revisit the lecture notes for the formula. The building blocks you need: `np.dot()` computes the dot product, `np.linalg.norm()` computes the length of a vector.

In [ ]:
def cosine_similarity(x, y):
    """Return the cosine similarity between two 1-D arrays."""
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))


def cosine_distance(x, y):
    """Return the cosine distance (1 - similarity)."""
    return 1 - cosine_similarity(x, y)


check_cosine_sim(cosine_similarity, x, y)
check_cosine_dist(cosine_distance, x, y)

### 🔍 Mahalanobis Distance

> The lecture states that the inverse covariance matrix gives "more weight to dimensions with less variance and less weight to dimensions with more variance." Why should a large spread in one dimension make a given displacement along that dimension less significant?

<details><summary>Thought</summary>

If a feature naturally varies a lot across the dataset, a moderate displacement along that axis is common and therefore less informative. The inverse covariance downscales such directions: a step of 10 along a dimension with variance 100 is only 1 standard deviation, while the same step along a dimension with variance 4 is 5 standard deviations.
</details>

We compute the sample covariance matrix from the coffee features. Unlike the diagonal matrix from the lecture, this one has off-diagonal entries that capture correlations between features (e.g. acidity and bitterness are negatively correlated).

Useful operations:
- `np.linalg.inv(M)` inverts a matrix
- `delta @ M @ delta` chains matrix multiplications (`@` is the matrix multiply operator)

In [ ]:
# Sample covariance matrix of the coffee features
Sigma = np.cov(features, rowvar=False)
print('Covariance matrix:')
print(Sigma.round(3))
print()


def mahalanobis_distance(x, y, cov):
    """Return the Mahalanobis distance given a covariance matrix."""
    delta = x - y
    cov_inv = np.linalg.inv(cov)
    return np.sqrt(delta @ cov_inv @ delta)


check_mahalanobis(mahalanobis_distance, x, y, Sigma)

### ⚖️ Compare All Four

> The four metrics map the same pair of coffee samples to very different numbers. Cosine says cold brew and espresso are quite close (high brew strength pulls both vectors in a similar direction), while Manhattan sees a large gap. Which answer is right?

<details><summary>Thought</summary>

Neither answer is "right" in isolation. Each metric encodes different assumptions about what similarity means. Cosine ignores magnitude and only sees the angle; Manhattan sums axis-aligned steps. The distance value is meaningless without knowing which metric produced it and what that metric cares about. Choosing a metric is a modelling decision.
</details>

Run the cell below to see all distances side by side.

In [ ]:
print('Distance between S01 (cold brew) and S12 (espresso)')
check_euclidean(euclidean_distance, x, y)
check_manhattan(manhattan_distance, x, y)
check_cosine_dist(cosine_distance, x, y)
check_mahalanobis(mahalanobis_distance, x, y, Sigma)


## Normalization & Cluster Visibility

The raw-feature heatmaps showed that brew strength dominates the distance. Let's fix that with normalization and see how the cluster structure changes.

### 📊 Z-Score Normalization

> The lecture states: "standardization does not change the data, it changes the expanse and spread of the space you measure it in. Units disappear." After this transformation, a distance of 1 means "one standard deviation apart." What information about the original physical units is lost, and why is that acceptable? What does this open up in terms of feature engineering? 

<details><summary>Thought</summary>

You lose the original scale: you can no longer tell whether a distance of 2 corresponds to 2 pH units or 2 mg/mL. What you gain is comparability: every feature now contributes on equal footing, measured in "how unusual is this difference given the feature's natural spread." and second you can now weigh features based on an importance you assign to them based on context.
</details>

Revisit the lecture notes for the formula.

Useful operations: `np.mean(data, axis=0)` computes the mean of each column, `np.std(data, axis=0)` the standard deviation.

In [ ]:
def zscore_normalize(data):
    """Return z-score normalized data (mean=0, std=1 per feature)."""
    return (data - np.mean(data, axis=0)) / np.std(data, axis=0)


features_z = zscore_normalize(features)
check_zscore(features_z, features)

### 📐 Min-Max Scaling

> The lecture warns that min-max scaling is "even more sensitive to outliers" than z-score, because a single extreme value stretches the entire scale. If one coffee sample had a brew strength of 50 mg/mL (double the next highest), how would that affect the scaling for all other samples on that feature?

<details><summary>Thought</summary>

The range would jump from ~17 to ~43, and all non-outlier samples would be compressed into roughly the lower half of [0, 1]. Most of the [0, 1] interval would be wasted on empty space between the outlier and the rest, reducing the effective resolution of that feature. Which will result in a sparse feature space, which makes the feature loose its expressiveness.
</details>

Revisit the lecture notes for the formula.

Useful operations: `np.min(data, axis=0)`, `np.max(data, axis=0)`.

In [ ]:
def minmax_normalize(data):
    """Return min-max scaled data (each feature in [0, 1])."""
    return (data - np.min(data, axis=0)) / (np.max(data, axis=0) - np.min(data, axis=0))


features_mm = minmax_normalize(features)
check_minmax(features_mm, features)

### 💡 Before vs. After — Scatter Plots

> After normalization, the axes no longer represent physical units (pH, mg/mL) but abstract standard-deviation units. In the raw scatter plots, cold brew and espresso overlap on the brew-strength axis. Does that overlap persist after normalization, or does a different feature separate them?

Let's see how the point cloud changes in 2D projections.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

feature_names_raw = ['Acidity (pH)', 'Bitterness', 'Brew Strength (mg/mL)']
feature_names_z = ['Acidity (z)', 'Bitterness (z)', 'Brew Strength (z)']

# Top row: raw features
axes[0, 0].scatter(features[:, 0], features[:, 1], c=colors, s=20, linewidths=0)
axes[0, 0].set_xlabel(feature_names_raw[0])
axes[0, 0].set_ylabel(feature_names_raw[1])
axes[0, 0].set_title('Raw: Acidity vs. Bitterness', fontsize=10)
tufte_axis(axes[0, 0])

axes[0, 1].scatter(features[:, 0], features[:, 2], c=colors, s=20, linewidths=0)
axes[0, 1].set_xlabel(feature_names_raw[0])
axes[0, 1].set_ylabel(feature_names_raw[2])
axes[0, 1].set_title('Raw: Acidity vs. Brew Strength', fontsize=10)
tufte_axis(axes[0, 1])

# Bottom row: z-score normalized
if features_z is not None:
    axes[1, 0].scatter(features_z[:, 0], features_z[:, 1], c=colors, s=20, linewidths=0)
    axes[1, 0].set_xlabel(feature_names_z[0])
    axes[1, 0].set_ylabel(feature_names_z[1])
    axes[1, 0].set_title('Z-Score: Acidity vs. Bitterness', fontsize=10)
    tufte_axis(axes[1, 0])

    axes[1, 1].scatter(features_z[:, 0], features_z[:, 2], c=colors, s=20, linewidths=0)
    axes[1, 1].set_xlabel(feature_names_z[0])
    axes[1, 1].set_ylabel(feature_names_z[2])
    axes[1, 1].set_title('Z-Score: Acidity vs. Brew Strength', fontsize=10)
    tufte_axis(axes[1, 1])
else:
    axes[1, 0].set_title('Z-Score: Acidity vs. Bitterness (not implemented)', fontsize=10)
    axes[1, 0].set_xlabel(feature_names_z[0])
    axes[1, 0].set_ylabel(feature_names_z[1])
    tufte_axis(axes[1, 0])
    axes[1, 1].set_title('Z-Score: Acidity vs. Brew Strength (not implemented)', fontsize=10)
    axes[1, 1].set_xlabel(feature_names_z[0])
    axes[1, 1].set_ylabel(feature_names_z[2])
    tufte_axis(axes[1, 1])

# Legend
for variety, color in variety_colors.items():
    axes[1, 1].scatter([], [], c=color, s=20, label=variety)
axes[1, 1].legend(frameon=False, fontsize=9)

plt.tight_layout()
plt.show()

### 🏁 Recap

**What we did:**
- ☕ Started with real coffee lab data and saw the scale differences.
- 📏🚕🧭🔍 Implemented four distance metrics from scratch and compared them on the same pair of samples.
- 🗺️ Explored the full dataset through heatmaps and saw how brew strength dominated raw distances.
- 💡 Applied normalization and watched the four variety clusters emerge.


**Now Head Back for Self-Check Questions and Key Learnings in the Lecture Notes** 


## Take It from Here — Next Steps (Optional)

The exercises below are **optional** extensions. They deepen your intuition but are not required to follow the rest of the course. Work through them at your own pace after the session.


### 🌍 Visualizing Distance Contours

> The Mahalanobis contour is an ellipse because the covariance stretches space unevenly. What would the Mahalanobis contour collapse to if all features had equal variance and zero correlation?

<details><summary>Thought</summary>

If the covariance matrix is a scalar multiple of the identity (equal variances, no correlations), the inverse covariance simply rescales all directions equally. The ellipse becomes a circle, and Mahalanobis reduces to a scaled version of Euclidean distance.
</details>

The plotting code below uses a 2D covariance matrix from the lecture (variance 9 in x, variance 49 in y) so we can draw contours on a flat grid. Your distance implementations from above handle any dimensionality, so they work here unchanged.

In [ ]:
# Build a grid of points around the origin
xx, yy = np.meshgrid(np.linspace(-2, 2, 400), np.linspace(-2, 2, 400))
origin = np.array([0.0, 0.0])

# 2D covariance from the lecture for visualization
Sigma_2d = np.array([[9, 0],
                     [0, 49]])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

dist_funcs = [
    ('Euclidean', lambda p: euclidean_distance(p, origin)),
    ('Manhattan', lambda p: manhattan_distance(p, origin)),
    ('Mahalanobis', lambda p: mahalanobis_distance(p, origin, Sigma_2d)),
]

for ax, (title, fn) in zip(axes, dist_funcs):
    test_val = fn(np.array([1.0, 1.0]))
    if test_val is None:
        ax.set_title(f'{title} (not implemented)', fontsize=10)
        ax.set_aspect('equal')
        ax.set_xlim(-2, 2)
        ax.set_ylim(-2, 2)
        tufte_axis(ax)
        continue

    Z = np.array([[fn(np.array([xi, yi]))
                   for xi in xx[0]] for yi in yy[:, 0]])
    ax.contour(xx, yy, Z, levels=[0.5, 1.0, 1.5], colors='black', linewidths=0.8)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.plot(0, 0, 'k+', markersize=8)
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    tufte_axis(ax)

plt.tight_layout()
plt.show()

**Observe:**
- Euclidean contours are **circles** — all directions are treated equally.
- Manhattan contours are **diamonds** — distance is measured along axes.
- Mahalanobis contours are **ellipses** — stretched along the axis with larger variance (here y), because a step in that direction is "less surprising" given the data's spread.

> Why is there no cosine contour? Cosine distance depends on the **angle** from the origin, not the distance to it. Points on the same ray all have cosine distance 0 to each other — the contours would be radial lines, not closed curves. Cosine measures direction, not position.


### 🎯 Find the Most Similar Pair

> If brew strength dominates the raw Euclidean distance, the closest pair will likely share similar brew strength. But does similar brew strength actually imply the same coffee type? Think about whether the answer you find is meaningful or an artifact of the scale mismatch.

Using the Euclidean distance matrix on raw features, find the pair of samples with the **smallest** non-zero distance. Check the result against the scatter plots.

In [ ]:
# Find the most similar pair (smallest non-zero distance)
D_copy = D_euclid.copy()
np.fill_diagonal(D_copy, np.inf)
min_idx = np.argmin(D_copy)
i, j = np.unravel_index(min_idx, D_copy.shape)
print(f'Most similar pair: {sample_ids[i]} ({labels[i]}) and {sample_ids[j]} ({labels[j]})')
print(f'Euclidean distance: {D_copy[i, j]:.4f}')

### 🧬 Covariance Matrix

> In the lecture, the coffee quality lab example shows that sourness and bitterness have a strong negative covariance. If two features are negatively correlated, the data cloud is elongated along a diagonal. How would Euclidean distance, which measures along the coordinate axes and not along the cloud, misjudge distances in that case?

<details><summary>Thought</summary>

Two points that lie along the elongated diagonal of the cloud might have large coordinate differences on both axes but are actually "typical" of the data's spread. Euclidean distance would call them far apart, while Mahalanobis, which aligns with the cloud's shape, would recognize the displacement as unremarkable. Conversely, a small step perpendicular to the cloud is highly unusual, but Euclidean treats it the same as any other small step.
</details>

Compute the sample covariance matrix of the **z-score normalized** data.

In [ ]:
# Compute the sample covariance matrix
# np.cov expects each ROW to be a variable, so we transpose.
if features_z is not None:
    cov_matrix = np.cov(features_z, rowvar=False)

    print("Covariance matrix (z-score normalized features):")
    print()

    # Pretty-print with feature names
    feature_names = ["acidity", "bitterness", "brew_str"]
    header = "            " + "  ".join(f"{n:>10}" for n in feature_names)
    print(header)
    for i, name in enumerate(feature_names):
        row = "  ".join(f"{cov_matrix[i, j]:10.3f}" for j in range(3))
        print(f"{name:>10}  {row}")
else:
    print("⬜ Covariance matrix: implement zscore_normalize first")

**Interpret the covariance matrix:**
- Diagonal entries are all close to 1 — that's what z-score normalization does.
- **Acidity and bitterness** have a strong negative covariance — low-pH coffees (espresso) tend to be more bitter. This makes sense.
- **Brew strength** has weaker correlations with the other two features.

This is exactly the kind of structure that Mahalanobis distance accounts for: it stretches the space to compensate for these correlations, so that "similar" means something more meaningful than raw Euclidean distance can capture.
